# An-Ra Unified Colab Trainer

Single end-to-end workflow. No Gemini/API keys required.


In [ ]:
# Cell 1: Environment + Drive
import os, sys, subprocess
from pathlib import Path
import torch
from google.colab import drive

drive.mount('/content/drive')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable T4 GPU in Colab Runtime settings')

# Install critical deps that must exist before cloning/importing
for dep in ['torch', 'fastapi', 'uvicorn', 'httpx', 'sympy', 'psutil', 'sentence-transformers']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', dep], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/KellerJordan/Muon'], check=False)


In [ ]:
# Cell 2: Clone repo + install ALL deps + path bootstrap
REPO = Path('/content/An-Ra')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=False)

# ── Install ALL project dependencies from requirements.txt ──
req_file = REPO / 'requirements.txt'
if req_file.exists():
    proc = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)],
        capture_output=True
    )
    if proc.returncode != 0:
        # Safe .decode() — never crash on NoneType
        err_out = (proc.stdout or b'').decode('utf-8', errors='replace')
        err_err = (proc.stderr or b'').decode('utf-8', errors='replace')
        print('WARNING: pip install -r requirements.txt returned non-zero.')
        print('STDOUT:', err_out[-2000:] if err_out else '(empty)')
        print('STDERR:', err_err[-2000:] if err_err else '(empty)')
        print('Continuing anyway — some optional deps may be missing.')
    else:
        print('All requirements.txt dependencies installed successfully.')
else:
    print('WARNING: requirements.txt not found in repo. Relying on Cell 1 deps.')

# ── Set ANRA_REPO_ROOT env var for legacy compatibility ──
os.environ['ANRA_REPO_ROOT'] = str(REPO)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
from anra_paths import inject_all_paths, ensure_dirs
inject_all_paths()
ensure_dirs()
print('Repo ready:', REPO)
print('ANRA_REPO_ROOT =', os.environ.get('ANRA_REPO_ROOT', '(NOT SET)'))


In [ ]:
# Cell 3: Ensure tokenizer.pkl exists (auto-create if missing)
import pickle
from anra_paths import TOKENIZER_DIR, TRAINING_DATA_DIR, DRIVE_DIR, get_tokenizer_file

tok_path = get_tokenizer_file()
if tok_path.exists():
    print('Tokenizer already exists:', tok_path)
else:
    print('tokenizer.pkl not found — creating from training data...')
    # Attempt to build from dataset
    dataset_candidates = [
        TRAINING_DATA_DIR / 'anra_dataset_v6_1.txt',
        REPO / 'anra_dataset_v6_1.txt',
        DRIVE_DIR / 'anra_dataset_v6_1.txt',
    ]
    dataset_file = None
    for c in dataset_candidates:
        if c.exists():
            dataset_file = c
            break
    if dataset_file is None:
        print('WARNING: No training dataset found. Creating minimal tokenizer.')
        corpus_text = 'H: Hello ANRA: I am An-Ra.'
    else:
        corpus_text = dataset_file.read_text(encoding='utf-8', errors='replace')
        print(f'Building tokenizer from {dataset_file} ({len(corpus_text)} chars)')

    sys.path.insert(0, str(TOKENIZER_DIR))
    from char_tokenizer import CharTokenizer
    tokenizer = CharTokenizer(corpus_text)
    TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
    out_path = TOKENIZER_DIR / 'tokenizer.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump(tokenizer, f)
    print(f'Tokenizer created: {out_path} (vocab_size={tokenizer.vocab_size})')
    # Also copy to Drive for persistence
    try:
        import shutil
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(out_path, DRIVE_DIR / 'tokenizer.pkl')
        print('Tokenizer backed up to Drive.')
    except Exception as e:
        print(f'Drive backup skipped: {e}')


In [ ]:
# Cell 4: Health check
import importlib
# Re-inject paths to be safe (idempotent)
from anra_paths import inject_all_paths
inject_all_paths()

mods = ['generate', 'app', 'symbolic_bridge', 'ouroboros_numpy', 'sovereignty_bridge', 'identity_injector']
for m in mods:
    try:
        importlib.import_module(m)
        print('OK', m)
    except Exception as e:
        print('WARN', m, e)


In [ ]:
# Cell 5: 30-minute resumable training session (single command)
# Auto-uses your dataset in training/training_data and resumes from local/Drive checkpoint.
cmd = [
    sys.executable, '-m', 'training.train_unified',
    '--mode', 'session',
    '--session_minutes', '30',
    '--checkpoint_path', 'anra_brain.pt',
    '--ouroboros_steps', '5000'
]
print('Running:', ' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('exit=', proc.returncode)


In [ ]:
# Cell 5B: 30-minute direct base-training routine (fallback)
import shutil
DRIVE = Path('/content/drive/MyDrive/AnRa')
for name in ['anra_ouroboros.pt', 'anra_brain_identity.pt', 'anra_brain.pt', 'tokenizer.pkl']:
    src = DRIVE / name
    dst = REPO / name
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print('restored', name)

result = subprocess.run([
    sys.executable, 'scripts/build_brain.py',
    '--data_path', 'training_data/anra_dataset_v6_1.txt',
    '--checkpoint_path', 'anra_brain.pt',
    '--max_minutes', '30'
], cwd=str(REPO), check=False)
print('session training exit=', result.returncode)


In [ ]:
# Cell 6: Verification
subprocess.run([sys.executable, 'scripts/verify_structure.py'], cwd=str(REPO), check=False)
subprocess.run([sys.executable, '-m', 'tests.test_suite'], cwd=str(REPO), check=False)
subprocess.run([sys.executable, 'anra.py', '--phase3-status'], cwd=str(REPO), check=False)
